In [ ]:
!pip install pretty_midi

In [ ]:
!unzip train.zip

In [1]:
import pretty_midi as pm

In [2]:
# extract notes from midi file

def extract_midi_notes(midi_file):
  # load midi file
  midi_data = pm.PrettyMIDI(midi_file)
  from pretty_midi.utilities import note_number_to_name

  note_sequence = []
  note_sequence_numeric = []

  for instrument in midi_data.instruments:
    if not instrument.is_drum:
      for note in instrument.notes:
        note_name = note_number_to_name(note.pitch)
        note_sequence.append(note_name)
        note_sequence_numeric.append(note.pitch)

  print("Extracted notes from midi-file: ", midi)
  print(note_sequence)
  print(note_sequence_numeric)

  return note_sequence, note_sequence_numeric

In [3]:
# generate sequences from MIDI files

import os
import numpy as np

input_x = []
output_y = []
seq_len = 7

# directory containing midi files to extract note sequences
for midi in os.listdir("train"):
  midi_file = os.path.join("train", midi)
  if os.path.isfile(midi_file):

    # extract midi notes
    note_sequence, note_sequence_numeric = extract_midi_notes(midi_file)

    for k in range(seq_len, len(note_sequence_numeric)):
      input_x.append(note_sequence_numeric[k-seq_len:k])
      output_y.append(note_sequence_numeric[k])

input_x = np.array(input_x)
output_y = np.array(output_y)

Extracted notes from midi-file:  waltz_09.mid
['D5', 'A4', 'F5', 'C#5', 'E5', 'A4', 'A5', 'D5', 'A#5', 'C#5', 'A5', 'C5', 'G5', 'A#4', 'D#5', 'A4', 'G4', 'A4', 'F#4', 'D5', 'A4', 'F5', 'C#5', 'E5', 'A4', 'A5', 'D5', 'A#5', 'C#5', 'A5', 'C5', 'G5', 'A#4', 'D#5', 'A4', 'G4', 'A4', 'F#4', 'D5', 'A#4', 'F5', 'A#4', 'D#5', 'G#4', 'G#5', 'G5', 'G5', 'A4', 'G5', 'A4', 'G5', 'C#5', 'A5', 'D5', 'D6', 'D5', 'A#5', 'A#4', 'G5', 'G4', 'E5', 'E4', 'C6', 'C5', 'A5', 'A4', 'F5', 'F4', 'D5', 'D4', 'A#5', 'D5', 'G#5', 'F4', 'A5', 'A4', 'F#5', 'E4', 'G5', 'A4', 'E5', 'F5', 'D5', 'G#4', 'C#5', 'A4', 'D5', 'A#4', 'F5', 'A#4', 'D#5', 'G#4', 'G#5', 'G5', 'G5', 'A4', 'G5', 'A4', 'G5', 'C#5', 'A5', 'D5', 'D6', 'D5', 'A#5', 'A#4', 'G5', 'G4', 'E5', 'E4', 'C6', 'C5', 'A5', 'A4', 'F5', 'F4', 'D5', 'D4', 'A#3', 'F5', 'F4', 'G#4', 'D4', 'E5', 'E4', 'G4', 'A3', 'D5', 'D4', 'F4', 'A#4', 'E4', 'D4', 'E4', 'C#4', 'C5', 'A4', 'F5', 'C#5', 'E5', 'A4', 'A5', 'D5', 'A#5', 'C#5', 'A5', 'C5', 'G5', 'A#4', 'D#5', 'A4', 'G4',

In [4]:
input_x.shape

(11032, 7)

In [8]:
# define LSTM model architecture

import keras
from keras.models import Sequential
from keras.layers import Input, Embedding, Dense, Dropout, LSTM

model = Sequential()
model.add(Input(shape=(seq_len, )))

# input_dim is vocabulary size which is all possible 128 notes
model.add(Embedding(input_dim=128, output_dim=32))

model.add(LSTM(128, return_sequences=True))
model.add(Dropout(0.3))
model.add(LSTM(128))
model.add(Dropout(0.3))

model.add(Dense(128, activation="softmax"))


In [9]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

In [ ]:
# train model on input sequences to predict next note
model.fit(input_x, output_y, epochs=100)

In [11]:
# generate music by predicting next k notes
# from given prompt sequence

def generate_sequence(prompt, model, k, seq_len):

  # convert prompt to numeric sequence
  from pretty_midi.utilities import note_name_to_number
  prompt_numeric = []
  for key in prompt:
    prompt_numeric.append(note_name_to_number(key))

  # generate k notes
  for i in range(k):
    # pass only sequence of expected length to model
    sequence = prompt_numeric[-seq_len:]

    # predict next note
    pred_y = model.predict(np.array([sequence]), verbose=0)
    y = np.argmax(pred_y)
    prompt_numeric.append(y)

  return prompt_numeric

In [12]:
prompt = ['D4', 'E4', 'F4', 'A4', 'B4', 'C5']
new_sequence = generate_sequence(prompt, model, 15, seq_len)

In [13]:
# get note names for new sequence
def sequence_to_notes(sequence):
  from pretty_midi.utilities import note_number_to_name
  note_sequence = []
  for key in sequence:
    note_sequence.append(note_number_to_name(key))
  return note_sequence

In [14]:
note_sequence = sequence_to_notes(new_sequence)
print(note_sequence)

['D4', 'E4', 'F4', 'A4', 'B4', 'C5', 'A4', 'A4', 'G#4', 'A4', 'F#4', 'D4', 'F#4', 'E4', 'F#4', 'F#4', 'E4', 'D#4', 'B4', 'E4', 'C#5']


In [15]:
# generate MIDI from new note sequence

new_melody = pm.PrettyMIDI()
new_instrument = pm.Instrument(program=74)

end_time = 0
duration = 0.7

for k in range(len(new_sequence)):
  note_number = new_sequence[k]
  start_time = end_time
  end_time = start_time + duration

  new_note = pm.Note(velocity=100, pitch=note_number, start=start_time, end=end_time)
  new_instrument.notes.append(new_note)

new_melody.instruments.append(new_instrument)
new_melody.write("output.mid")